# Dynamic Arrays

```{contents}
:local:
:depth: 2
```

A dynamic array gives the programmer the feel of a growable sequence while storing values in ordinary contiguous array storage behind the scenes.


```{index} contiguous storage
```
```{index} dynamic array
```

## Contiguous Storage


An array stores elements in numbered positions. If the array has five slots, valid indexes are `0` through `4`.

```text
index:  0   1   2   3   4
value: [10][20][30][  ][  ]
```

The array has a fixed **capacity**: the number of slots available. A dynamic array also tracks **count**: the number of slots currently being used.


In [ ]:
int[] storage = new int[5];
int count = 0;

storage[count] = 10;
count++;
storage[count] = 20;
count++;
storage[count] = 30;
count++;

Console.WriteLine($"count={count}, capacity={storage.Length}");
Console.WriteLine(storage[1]);


```{index} capacity
```
```{index} resizing; dynamic array
```

## Resizing by Copying


When the array fills up, the dynamic array cannot stretch the existing array. It allocates a larger array, copies the existing values, then continues adding into the new storage.

A common strategy is to double the capacity each time resizing is needed. This makes individual resize steps expensive, but keeps most `Add` operations cheap.


In [ ]:
static int[] Grow(int[] oldStorage)
{
    int[] larger = new int[oldStorage.Length * 2];
    for (int i = 0; i < oldStorage.Length; i++)
    {
        larger[i] = oldStorage[i];
    }
    return larger;
}

int[] numbers = new int[] { 10, 20 };
numbers = Grow(numbers);
numbers[2] = 30;

Console.WriteLine(string.Join(", ", numbers));


```{index} indexed access
```

## Worked Example: A Small Dynamic Array


This class stores integers in an internal array. `Add` appends to the next free slot. The indexer reads an existing value by index.


In [ ]:
var values = new DynamicIntArray();
for (int value = 10; value <= 50; value += 10)
{
    values.Add(value);
    Console.WriteLine($"added={value}, count={values.Count}, capacity={values.Capacity}");
}

Console.WriteLine($"values[3]={values[3]}");

public class DynamicIntArray
{
    private int[] items = new int[2];
    public int Count { get; private set; }
    public int Capacity => items.Length;

    public void Add(int value)
    {
        if (Count == items.Length)
        {
            int[] larger = new int[items.Length * 2];
            Array.Copy(items, larger, items.Length);
            items = larger;
        }

        items[Count] = value;
        Count++;
    }

    public int this[int index]
    {
        get
        {
            if (index < 0 || index >= Count)
                throw new IndexOutOfRangeException();
            return items[index];
        }
    }
}


```{index} amortized cost
```

## Cost of Append and Indexing


Dynamic arrays have two different stories:

- Reading `items[index]` is O(1) because the index points directly to the slot.
- Most appends are O(1), but an append that triggers resizing is O(n) because it copies existing values.

Over many appends, doubling capacity gives **amortized O(1)** append cost. Amortized means the occasional expensive resize is spread across many cheap operations.


```{index} sequence; invariant; Count; Capacity
```

## A Sequence Contract and Its Invariants

A service desk keeps work orders in processing order. The values might be strings such as `WO-10` and `WO-30`, but the representation should work for other types too. Chapter 14 separated an ADT's behavior from its implementation. Here the behavior is an ordered sequence: append a work order, read its position, insert an urgent order, and remove a completed order.

For an array-backed implementation, three facts must remain true after every public operation:

1. `0 <= Count <= Capacity`.
2. The live sequence occupies indexes `0` through `Count - 1` without gaps.
3. Values outside that range are storage, not members of the sequence.

A zero in an integer array is not a reliable empty marker: zero might be a real value. `Count` determines membership. With `Count = 3` and `Capacity = 5`, reading index `4` must fail even though the underlying array physically has that slot. An insertion at index `3` is valid because it appends; a read at index `3` is invalid.

**Check your understanding (basic):** An array has capacity eight and count three. List the legal read indexes and the legal insertion indexes. Explain why these sets differ.

```{index} dynamic array; resize trace; alias
```

## Trace Growth Before Writing Code

Start with capacity two. Appending `WO-10` and `WO-30` uses both slots. Before a third append, allocate four slots, copy the two live references, and replace the private storage reference. The logical sequence has not changed yet. Only then write the new value and increase `Count`.

```text
Before growth: items -> [WO-10][WO-30]                 Count = 2
New storage:   larger -> [WO-10][WO-30][unused][unused]
After growth:  items -> [WO-10][WO-30][unused][unused]   Count = 2
After append:  items -> [WO-10][WO-30][WO-40][unused]    Count = 3
```

For reference-type elements, copying the array copies references, not the referenced objects. The work order objects would retain their identity. Our string labels let you concentrate on the sequence rather than object mutation.

| Append number | Count after append | Capacity | Existing entries copied |
| ---: | ---: | ---: | ---: |
| 1 | 1 | 2 | 0 |
| 2 | 2 | 2 | 0 |
| 3 | 3 | 4 | 2 |
| 4 | 4 | 4 | 0 |
| 5 | 5 | 8 | 4 |
| 6 | 6 | 8 | 0 |
| 7 | 7 | 8 | 0 |
| 8 | 8 | 8 | 0 |
| 9 | 9 | 16 | 8 |

**Predict (basic):** Before reading on, give the capacity and copy count for append 10. Does it resize again?

```{index} dynamic array; insertion; removal; shifting
```

## Middle Updates: Preserve the Unread Values

Suppose the live values are `[10, 20, 40, 50]`, and you must insert `30` at index two. First ensure space for a fifth value. Then shift from the right end toward the insertion point:

| Step | Live values and destination slot | Reason |
| --- | --- | --- |
| Start | `[10, 20, 40, 50, unused]` | Index two must become available. |
| Copy index 3 to 4 | `[10, 20, 40, 50, 50]` | Save the last value first. |
| Copy index 2 to 3 | `[10, 20, 40, 40, 50]` | The old index three is already safe. |
| Write 30 at index 2 | `[10, 20, 30, 40, 50]` | Increase count after storing the new value. |

Shifting left to right would overwrite values you have not copied yet. The temporary duplicates in the trace are expected; the sequence invariant applies at the operation boundary.

Removal moves in the opposite direction. To remove index one from `[10, 20, 30, 40]`, save `20`, copy `30` left, then copy `40` left. Reduce count to three and clear the formerly live final slot. Clearing prevents a removed reference from being retained unnecessarily; it does not reduce capacity.

Insertion at index `i` moves `n - i` existing entries. Removal moves `n - i - 1`. Appending and removing the last item move no entries, although an append may still resize.

**Trace (applied):** Insert at index zero and then remove the last item. Record each assignment. Explain why reversing the insertion loop corrupts the original sequence.

```{index} generics; dynamic array; bounds checking
```

## Complete Generic Implementation

The following implementation gives the work-order sequence a reusable element type `T`. `EnsureRoom` handles physical storage; `Insert` handles logical order. Keeping these responsibilities separate means growth cannot accidentally change count.

`default!` clears the unused slot. The exclamation mark suppresses a nullable-analysis warning for that storage assignment; it does not check anything at runtime. Clients cannot read the cleared slot because the indexer checks `Count`.

Run this cell independently. Expect count three, capacity four, the labels in order, and then removal of `WO-10`. This is an instructional implementation: it omits shrinking, iterators, concurrency, and protection against enormous capacity overflow.

In [ ]:
using System;
var orders = new WorkOrderArray<string>();
orders.Add("WO-10");
orders.Add("WO-30");
orders.Insert(1, "WO-20");
Console.WriteLine($"count={orders.Count}, capacity={orders.Capacity}");
for (int i = 0; i < orders.Count; i++) Console.WriteLine(orders[i]);
Console.WriteLine($"removed={orders.RemoveAt(0)}");
for (int i = 0; i < orders.Count; i++) Console.WriteLine(orders[i]);
public class WorkOrderArray<T>
{
    private T[] items = new T[2];
    public int Count { get; private set; }
    public int Capacity => items.Length;
    public T this[int index]
    {
        get { CheckElement(index); return items[index]; }
    }
    private void CheckElement(int index)
    {
        if (index < 0 || index >= Count)
            throw new ArgumentOutOfRangeException(nameof(index));
    }
    private void EnsureRoom()
    {
        if (Count < Capacity) return;
        var larger = new T[Capacity * 2];
        Array.Copy(items, larger, Count);
        items = larger;
    }
    public void Add(T value) => Insert(Count, value);
    public void Insert(int index, T value)
    {
        if (index < 0 || index > Count)
            throw new ArgumentOutOfRangeException(nameof(index));
        EnsureRoom();
        for (int i = Count; i > index; i--)
            items[i] = items[i - 1];
        items[index] = value;
        Count++;
    }
    public T RemoveAt(int index)
    {
        CheckElement(index);
        T removed = items[index];
        for (int i = index; i < Count - 1; i++)
            items[i] = items[i + 1];
        Count--;
        items[Count] = default!;
        return removed;
    }
}

Read the implementation in operation order. `Insert` validates before mutation, grows before shifting, moves right to left, writes the new value, and increments count once. `RemoveAt` validates before mutation, saves the result, compacts left to right, decreases count once, and clears the unused slot.

**Modify (applied):** Add `Contains(T value)` using `EqualityComparer<T>.Default`. Test repeated values and a missing value. Explain why supporting generics does not make searching constant-time.

```{index} amortized analysis; worst case; dynamic array
```

## Why Doubling Gives Amortized Constant Cost

For nine appends, the trace copies `2 + 4 + 8 = 14` existing entries and writes nine new entries. Counting these array writes gives 23 writes across nine operations. Some individual appends are costly, but the total grows in proportion to the number of appends.

For a longer sequence, copy sizes form `2 + 4 + 8 + ...`. Each term is twice the previous one, so the sum stays below twice the final number of inserted items. Adding one write per appended item still gives a total proportional to `n`. Dividing total work by `n` gives a constant amortized cost. This is a bound over an operation sequence, not an assumption about random inputs.

Growing by one slot each time would copy `2 + 3 + 4 + ... + (n - 1)` entries. That total grows quadratically. Avoiding spare capacity would therefore make repeated appends progressively expensive.

The tradeoff is spare storage. Doubling leaves unused slots after growth, and a resize temporarily keeps both old and new arrays alive. Amortized O(1) also does not guarantee a short pause for every append. A system with a strict response deadline may reserve capacity ahead of time.

**Explain (challenge):** Compare doubling with adding one slot using ten appends starting at capacity two. Count copies, not elapsed milliseconds. State both the worst-case cost of one append and the amortized cost of the whole sequence.

```{index} practice
```

## Practice


1. Trace the capacities after appending six values to a dynamic array that starts with capacity `2` and doubles when full.
2. Explain why indexed access is fast in an array-backed representation.
3. Add a `Contains(int value)` method to `DynamicIntArray`. What is its worst-case cost?


```{index} summary
```

## Summary


A dynamic array combines contiguous storage with a resizing policy. It is strong when a program needs fast indexing and frequent appends, but middle insertion and deletion require shifting elements.


```{rubric} Footnotes
```
[^1]: C# `List<T>` is implemented as a dynamic array conceptually similar to this simplified example, with many additional features and optimizations.
